In [2]:
import torch
import torch.nn.functional as F
from torch import nn
import torch.utils.checkpoint

In [ ]:
gate = nn.Linear(512, 10)
logits_sample = torch.randn(512, 10)
print(logits_sample)
print("------------------------")
topk_logits_sample, indices = logits_sample.topk(5, dim=-1) # go through each row and select the top 5 values across the col and their indices
print(topk_logits_sample)
print("------------------------")
print(indices) 

tensor([[-0.1514, -0.2452,  0.7438,  ..., -2.2600, -0.6433,  0.7480],
        [ 0.2520, -0.7866,  0.6724,  ...,  0.2455, -0.2082,  0.1481],
        [-0.1892, -0.4734,  0.5137,  ...,  0.6093, -0.1392,  1.1558],
        ...,
        [ 0.5576,  0.6745,  0.3134,  ...,  1.1747,  0.3786,  0.6262],
        [ 1.2227,  0.8169,  1.0717,  ...,  1.0981, -0.1585, -0.8062],
        [-0.3926,  1.4992, -1.6746,  ..., -0.5472,  1.7818,  0.2271]])
------------------------
tensor([[ 1.0316,  0.7480,  0.7443,  0.7438,  0.0426],
        [ 2.1556,  0.6724,  0.2520,  0.2455,  0.1481],
        [ 1.1558,  1.0977,  0.8361,  0.6093,  0.5137],
        ...,
        [ 1.3112,  1.1747,  1.0741,  0.6745,  0.6262],
        [ 1.7102,  1.2227,  1.0981,  1.0717,  0.8169],
        [ 1.7818,  1.4992,  0.2271,  0.1795, -0.0190]])
------------------------
tensor([[3, 9, 6, 2, 5],
        [6, 2, 0, 7, 9],
        [9, 6, 4, 7, 2],
        ...,
        [5, 7, 6, 1, 9],
        [5, 0, 7, 2, 1],
        [8, 1, 9, 3, 6]])


In [11]:
zeros = torch.full_like(logits_sample, float("-inf")) 
zeros.shape

torch.Size([512, 10])

In [14]:
sparse_logits = zeros.scatter(dim=-1, index=indices, src=topk_logits_sample)  
sparse_logits

tensor([[  -inf,   -inf, 0.7438,  ...,   -inf,   -inf, 0.7480],
        [0.2520,   -inf, 0.6724,  ..., 0.2455,   -inf, 0.1481],
        [  -inf,   -inf, 0.5137,  ..., 0.6093,   -inf, 1.1558],
        ...,
        [  -inf, 0.6745,   -inf,  ..., 1.1747,   -inf, 0.6262],
        [1.2227, 0.8169, 1.0717,  ..., 1.0981,   -inf,   -inf],
        [  -inf, 1.4992,   -inf,  ...,   -inf, 1.7818, 0.2271]])

In [15]:
sparse_probs = F.softmax(sparse_logits, dim=-1) 
sparse_probs

tensor([[0.0000, 0.0000, 0.2069,  ..., 0.0000, 0.0000, 0.2077],
        [0.0899, 0.0000, 0.1368,  ..., 0.0893, 0.0000, 0.0810],
        [0.0000, 0.0000, 0.1394,  ..., 0.1534, 0.0000, 0.2649],
        ...,
        [0.0000, 0.1432, 0.0000,  ..., 0.2361, 0.0000, 0.1364],
        [0.1985, 0.1323, 0.1707,  ..., 0.1753, 0.0000, 0.0000],
        [0.0000, 0.3233, 0.0000,  ..., 0.0000, 0.4289, 0.0906]])

In [17]:
sparse_probs.shape

torch.Size([512, 10])

In [ ]:
gate_logit = logits_sample.view(-1, 10)
gate_logit

tensor([[-0.1514, -0.2452,  0.7438,  ..., -2.2600, -0.6433,  0.7480],
        [ 0.2520, -0.7866,  0.6724,  ...,  0.2455, -0.2082,  0.1481],
        [-0.1892, -0.4734,  0.5137,  ...,  0.6093, -0.1392,  1.1558],
        ...,
        [ 0.5576,  0.6745,  0.3134,  ...,  1.1747,  0.3786,  0.6262],
        [ 1.2227,  0.8169,  1.0717,  ...,  1.0981, -0.1585, -0.8062],
        [-0.3926,  1.4992, -1.6746,  ..., -0.5472,  1.7818,  0.2271]])

In [22]:
torch.topk(sparse_probs, 5, dim=-1)

torch.return_types.topk(
values=tensor([[0.2758, 0.2077, 0.2070, 0.2069, 0.1026],
        [0.6030, 0.1368, 0.0899, 0.0893, 0.0810],
        [0.2649, 0.2499, 0.1924, 0.1534, 0.1394],
        ...,
        [0.2707, 0.2361, 0.2135, 0.1432, 0.1364],
        [0.3232, 0.1985, 0.1753, 0.1707, 0.1323],
        [0.4289, 0.3233, 0.0906, 0.0864, 0.0708]]),
indices=tensor([[3, 9, 6, 2, 5],
        [6, 2, 0, 7, 9],
        [9, 6, 4, 7, 2],
        ...,
        [5, 7, 6, 1, 9],
        [5, 0, 7, 2, 1],
        [8, 1, 9, 3, 6]]))

In [24]:
expert_mask = torch.nn.functional.one_hot(indices, 10)
expert_mask

tensor([[[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 1],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 1,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 1,  ..., 0, 0, 0],
         [1, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 1, 0, 0],
         [0, 0, 0,  ..., 0, 0, 1]],

        [[0, 0, 0,  ..., 0, 0, 1],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 1, 0, 0],
         [0, 0, 1,  ..., 0, 0, 0]],

        ...,

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 1, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 1, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 1]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [1, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 1, 0, 0],
         [0, 0, 1,  ..., 0, 0, 0],
         [0, 1, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 1, 0],
         [0, 1, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 

In [30]:
expert_mask.shape

torch.Size([512, 5, 10])

In [ ]:
# version 1
tokens_per_expert = torch.mean(expert_mask.float(), dim=0)
tokens_per_expert

tensor([[0.0938, 0.1094, 0.0801, 0.0859, 0.0938, 0.0996, 0.1309, 0.0918, 0.1172,
         0.0977],
        [0.0977, 0.0957, 0.0859, 0.1250, 0.0879, 0.1133, 0.1055, 0.0918, 0.0859,
         0.1113],
        [0.0918, 0.0820, 0.1191, 0.0957, 0.1230, 0.1113, 0.0957, 0.0996, 0.0723,
         0.1094],
        [0.0762, 0.1016, 0.1094, 0.1055, 0.0938, 0.0996, 0.1230, 0.0996, 0.1035,
         0.0879],
        [0.1035, 0.1055, 0.0938, 0.0879, 0.1113, 0.1016, 0.0801, 0.1113, 0.0977,
         0.1074]])

In [29]:
# version 2
tokens_per_expert = expert_mask.sum(dim=(0, 1)) / (expert_mask.shape[0] * expert_mask.shape[1])
tokens_per_expert

tensor([0.0926, 0.0988, 0.0977, 0.1000, 0.1020, 0.1051, 0.1070, 0.0988, 0.0953,
        0.1027])

In [31]:
tokens_per_expert.shape

torch.Size([10])

In [32]:
router_prob_per_expert = torch.mean(sparse_probs, dim=0)
router_prob_per_expert

tensor([0.0920, 0.0986, 0.0919, 0.0988, 0.1023, 0.1059, 0.1124, 0.0982, 0.0991,
        0.1007])